# 01 — Análise Exploratória do Dataset Spotify
**Projeto:** Técnicas de Indexação e Ranking em Busca de Músicas  
**Parte:** 1 — Recuperação Clássica (BM25)  
**Objetivo:** Entender a estrutura do dataset, identificar problemas de qualidade e documentar as colunas que serão usadas nas Partes 1, 2 e 3.

---

## 0. Imports e configuração

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
import warnings

warnings.filterwarnings('ignore')

# Estilo dos gráficos
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (12, 5)

print('Bibliotecas carregadas com sucesso.')

## 1. Carregamento do dataset

> **Instrução:** Após baixar o dataset do Kaggle (`lordpatil/spotify-metadata-by-annas-archive`), descompacte o arquivo e ajuste o caminho abaixo conforme necessário.

In [ ]:
# Ajuste o caminho para o arquivo CSV do dataset
DATASET_PATH = '../data/spotify_metadata.csv'

df = pd.read_csv(DATASET_PATH, low_memory=False)

print(f'Dataset carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
df.head(3)

## 2. Visão geral da estrutura

In [ ]:
# Tipos de dados e contagem de nulos
info = pd.DataFrame({
    'dtype'       : df.dtypes,
    'nulos'       : df.isnull().sum(),
    'nulos_%'     : (df.isnull().sum() / len(df) * 100).round(2),
    'únicos'      : df.nunique(),
    'exemplo'     : df.iloc[0]
})
info

In [ ]:
# Identificar colunas por categoria
TEXT_COLS    = [c for c in ['track_name', 'artist_name', 'album_name', 'genre', 'track_genre'] if c in df.columns]
AUDIO_COLS   = [c for c in ['acousticness','danceability','energy','valence',
                             'speechiness','instrumentalness','liveness'] if c in df.columns]
NUMERIC_COLS = [c for c in ['tempo','loudness','duration_ms','popularity',
                             'key','mode','time_signature'] if c in df.columns]

print('Colunas textuais  :', TEXT_COLS)
print('Atributos de áudio:', AUDIO_COLS)
print('Outros numéricos  :', NUMERIC_COLS)

## 3. Duplicatas

In [ ]:
n_dup = df.duplicated().sum()
print(f'Linhas duplicadas (100% idênticas): {n_dup:,}  ({n_dup/len(df)*100:.2f}%)')

# Duplicatas por track_name + artist_name (mesma música, metadados repetidos)
key_cols = [c for c in ['track_name', 'artist_name'] if c in df.columns]
if key_cols:
    n_dup_key = df.duplicated(subset=key_cols).sum()
    print(f'Duplicatas por ({" + ".join(key_cols)}): {n_dup_key:,}  ({n_dup_key/len(df)*100:.2f}%)')

## 4. Análise das colunas textuais

In [ ]:
# Comprimento médio dos campos textuais
for col in TEXT_COLS:
    lengths = df[col].dropna().astype(str).str.len()
    print(f'{col:20s} | média: {lengths.mean():.1f} chars | max: {lengths.max()} | nulos: {df[col].isnull().sum()}')

In [ ]:
# Top 15 artistas mais frequentes
if 'artist_name' in df.columns:
    top_artists = df['artist_name'].value_counts().head(15)
    fig, ax = plt.subplots(figsize=(12, 4))
    top_artists.plot(kind='barh', ax=ax, color=sns.color_palette('muted')[0])
    ax.invert_yaxis()
    ax.set_title('Top 15 artistas por número de faixas')
    ax.set_xlabel('Número de faixas')
    plt.tight_layout()
    plt.show()

In [ ]:
# Distribuição de comprimento do nome da faixa
if 'track_name' in df.columns:
    track_len = df['track_name'].dropna().astype(str).str.len()
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(track_len, bins=60, color=sns.color_palette('muted')[2], edgecolor='white')
    ax.set_title('Distribuição do comprimento de track_name (chars)')
    ax.set_xlabel('Caracteres')
    ax.set_ylabel('Frequência')
    plt.tight_layout()
    plt.show()
    print(f'Mediana: {track_len.median():.0f} chars | P95: {track_len.quantile(0.95):.0f} chars')

In [ ]:
# Termos mais comuns em track_name (pré-processamento superficial)
if 'track_name' in df.columns:
    STOPWORDS_MUSICA = {'feat', 'ft', 'remix', 'remaster', 'remastered',
                        'version', 'live', 'edit', 'mix', 'radio', 'original',
                        'the', 'a', 'an', 'and', 'of', 'in', 'is', 'it',
                        'to', 'you', 'i', 'my', 'me', 'with', 'on'}

    all_tokens = []
    for name in df['track_name'].dropna().astype(str):
        tokens = [t.lower().strip('()[]- ') for t in name.split()]
        all_tokens.extend([t for t in tokens if t and t not in STOPWORDS_MUSICA and len(t) > 2])

    top_tokens = Counter(all_tokens).most_common(20)
    words, counts = zip(*top_tokens)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(words, counts, color=sns.color_palette('muted')[4])
    ax.set_title('Top 20 termos mais comuns em track_name (sem stopwords musicais)')
    ax.set_ylabel('Frequência')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

## 5. Análise dos atributos de áudio

In [ ]:
# Estatísticas descritivas dos atributos de áudio
df[AUDIO_COLS].describe().round(3)

In [ ]:
# Distribuições dos atributos de áudio (0–1)
n = len(AUDIO_COLS)
cols_per_row = 4
rows = (n + cols_per_row - 1) // cols_per_row
fig, axes = plt.subplots(rows, cols_per_row, figsize=(14, 3.5 * rows))
axes = axes.flatten()

for i, col in enumerate(AUDIO_COLS):
    axes[i].hist(df[col].dropna(), bins=50,
                 color=sns.color_palette('muted')[i % 8], edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('Valor (0–1)')
    axes[i].set_ylabel('Frequência')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuição dos atributos de áudio', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlação entre atributos de áudio
all_num = AUDIO_COLS + [c for c in NUMERIC_COLS if c in df.columns and df[c].dtype in ['float64','int64']]

fig, ax = plt.subplots(figsize=(10, 8))
corr = df[all_num].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de correlação — atributos de áudio')
plt.tight_layout()
plt.show()

## 6. Análise de popularidade

In [ ]:
if 'popularity' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Distribuição de popularidade
    axes[0].hist(df['popularity'].dropna(), bins=50,
                 color=sns.color_palette('muted')[1], edgecolor='white')
    axes[0].set_title('Distribuição de popularidade')
    axes[0].set_xlabel('Popularidade (0–100)')
    axes[0].set_ylabel('Frequência')

    # Boxplot: popularidade por faixas de energy
    if 'energy' in df.columns:
        df['energy_bin'] = pd.cut(df['energy'], bins=5,
                                   labels=['Muito baixa','Baixa','Média','Alta','Muito alta'])
        df.boxplot(column='popularity', by='energy_bin', ax=axes[1])
        axes[1].set_title('Popularidade por nível de energy')
        axes[1].set_xlabel('Faixa de energy')
        axes[1].set_ylabel('Popularidade')
        plt.suptitle('')

    plt.tight_layout()
    plt.show()

    print(f"\nPopularidade — média: {df['popularity'].mean():.1f} | mediana: {df['popularity'].median():.1f}")
    print(f"Músicas com popularidade 0: {(df['popularity']==0).sum():,} ({(df['popularity']==0).sum()/len(df)*100:.1f}%)")

In [ ]:
# Top 10 músicas mais populares
if 'popularity' in df.columns:
    cols_show = [c for c in ['track_name', 'artist_name', 'album_name', 'popularity'] if c in df.columns]
    print('Top 10 músicas mais populares:')
    display(df[cols_show].sort_values('popularity', ascending=False).drop_duplicates('track_name').head(10).reset_index(drop=True))

## 7. Análise de tempo (tempo) e duração

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribuição de BPM (tempo)
if 'tempo' in df.columns:
    axes[0].hist(df['tempo'].dropna(), bins=60,
                 color=sns.color_palette('muted')[3], edgecolor='white')
    axes[0].set_title('Distribuição de BPM (tempo)')
    axes[0].set_xlabel('BPM')
    axes[0].set_ylabel('Frequência')
    axes[0].axvline(df['tempo'].median(), color='red', linestyle='--', label=f'Mediana: {df["tempo"].median():.0f} BPM')
    axes[0].legend()
else:
    axes[0].set_visible(False)

# Distribuição de duração (em minutos)
if 'duration_ms' in df.columns:
    dur_min = df['duration_ms'] / 60000
    dur_clipped = dur_min[dur_min < 15]  # ignora outliers
    axes[1].hist(dur_clipped, bins=60,
                 color=sns.color_palette('muted')[5], edgecolor='white')
    axes[1].set_title('Distribuição de duração das faixas')
    axes[1].set_xlabel('Duração (minutos)')
    axes[1].set_ylabel('Frequência')
    axes[1].axvline(dur_clipped.median(), color='red', linestyle='--',
                    label=f'Mediana: {dur_clipped.median():.1f} min')
    axes[1].legend()
else:
    axes[1].set_visible(False)

plt.tight_layout()
plt.show()

## 8. Análise de gêneros musicais

In [ ]:
genre_col = next((c for c in ['track_genre', 'genre'] if c in df.columns), None)

if genre_col:
    top_genres = df[genre_col].value_counts().head(20)
    fig, ax = plt.subplots(figsize=(12, 5))
    top_genres.plot(kind='bar', ax=ax, color=sns.color_palette('muted', 20))
    ax.set_title(f'Top 20 gêneros mais frequentes ({genre_col})')
    ax.set_xlabel('Gênero')
    ax.set_ylabel('Número de faixas')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()
    print(f'Total de gêneros únicos: {df[genre_col].nunique()}')
else:
    print('Nenhuma coluna de gênero encontrada no dataset.')

In [ ]:
# Perfil médio dos atributos de áudio por gênero (top 10 gêneros)
if genre_col and AUDIO_COLS:
    top10_genres = df[genre_col].value_counts().head(10).index
    genre_profile = df[df[genre_col].isin(top10_genres)].groupby(genre_col)[AUDIO_COLS].mean()

    fig, ax = plt.subplots(figsize=(14, 6))
    genre_profile.T.plot(kind='bar', ax=ax)
    ax.set_title('Perfil médio de atributos de áudio por gênero (top 10)')
    ax.set_ylabel('Valor médio')
    ax.set_xlabel('Atributo de áudio')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Gênero', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.show()

## 9. Scatter plots entre atributos relevantes

In [ ]:
# Pares relevantes para o projeto de RI
pairs = [
    ('energy', 'danceability'),
    ('energy', 'acousticness'),
    ('valence', 'danceability'),
]
pairs = [(a, b) for a, b in pairs if a in df.columns and b in df.columns]

fig, axes = plt.subplots(1, len(pairs), figsize=(5 * len(pairs), 4))
if len(pairs) == 1:
    axes = [axes]

sample = df.sample(min(3000, len(df)), random_state=42)
color_col = 'popularity' if 'popularity' in df.columns else None

for ax, (x, y) in zip(axes, pairs):
    scatter_kw = dict(alpha=0.3, s=10, edgecolors='none')
    if color_col:
        sc = ax.scatter(sample[x], sample[y], c=sample[color_col],
                        cmap='RdYlGn', **scatter_kw)
        plt.colorbar(sc, ax=ax, label='Popularidade')
    else:
        ax.scatter(sample[x], sample[y], **scatter_kw)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f'{x} × {y}')

plt.suptitle('Relações entre atributos de áudio (colorido por popularidade)', y=1.02)
plt.tight_layout()
plt.show()

## 10. Identificação de outliers e valores problemáticos

In [ ]:
print('=== Verificação de valores fora do intervalo esperado ===\n')

# Atributos de áudio devem estar entre 0 e 1
for col in AUDIO_COLS:
    out_range = ((df[col] < 0) | (df[col] > 1)).sum()
    if out_range > 0:
        print(f'  {col}: {out_range} valores fora de [0, 1]')
    else:
        print(f'  {col}: OK (todos em [0, 1])')

# Popularidade deve estar entre 0 e 100
if 'popularity' in df.columns:
    out_pop = ((df['popularity'] < 0) | (df['popularity'] > 100)).sum()
    print(f'  popularity: {out_pop} valores fora de [0, 100]')

# Tempo (BPM) — outliers extremos
if 'tempo' in df.columns:
    extreme_bpm = ((df['tempo'] < 30) | (df['tempo'] > 300)).sum()
    print(f'  tempo: {extreme_bpm} valores fora de [30, 300] BPM')

In [ ]:
# Boxplots para detectar outliers nos campos numéricos
numeric_for_box = [c for c in ['tempo', 'loudness', 'popularity'] if c in df.columns]

if numeric_for_box:
    fig, axes = plt.subplots(1, len(numeric_for_box), figsize=(4 * len(numeric_for_box), 4))
    if len(numeric_for_box) == 1:
        axes = [axes]
    for ax, col in zip(axes, numeric_for_box):
        ax.boxplot(df[col].dropna(), patch_artist=True,
                   boxprops=dict(facecolor=sns.color_palette('muted')[0], alpha=0.7))
        ax.set_title(col)
        ax.set_xticklabels([])
    plt.suptitle('Boxplots — campos numéricos')
    plt.tight_layout()
    plt.show()

## Extra: Leitura alternativa via Parquet

O dataset também está disponível em formato Parquet (`tracks.parquet`, ~256M registros).
Para datasets grandes demais para pandas, usar DuckDB:

```python
import duckdb
duckdb.sql("SELECT count(*) FROM '../data/tracks.parquet'")
```

> **Nota:** o arquivo Parquet completo tem ~256 milhões de linhas e pode causar erros de memória com `pd.read_parquet()`. Prefira DuckDB ou leitura em chunks.

## 11. Resumo e decisões para a limpeza

In [ ]:
print('=' * 60)
print('RESUMO DA EDA — DECISÕES PARA A PARTE 1 (BM25)')
print('=' * 60)

total = len(df)

print(f'\n1. TAMANHO DO DATASET')
print(f'   Total de registros: {total:,}')
print(f'   Total de colunas  : {df.shape[1]}')

print(f'\n2. PROBLEMAS IDENTIFICADOS')
print(f'   Duplicatas totais : {df.duplicated().sum():,}')
for col in TEXT_COLS:
    n = df[col].isnull().sum()
    if n > 0:
        print(f'   Nulos em {col:<20}: {n:,} ({n/total*100:.2f}%)')
for col in AUDIO_COLS:
    n = df[col].isnull().sum()
    if n > 0:
        print(f'   Nulos em {col:<20}: {n:,} ({n/total*100:.2f}%)')

print(f'\n3. COLUNAS PARA O CAMPO BM25 (text_field)')
print(f'   {" + ".join(TEXT_COLS)}')

print(f'\n4. FEATURES NUMÉRICAS PARA AS PARTES 2 E 3')
print(f'   {AUDIO_COLS + [c for c in NUMERIC_COLS if c in df.columns]}')

print(f'\n5. PRÓXIMOS PASSOS')
print(f'   [x] Remover duplicatas')
print(f'   [x] Tratar nulos nos campos textuais (preencher com string vazia)')
print(f'   [x] Normalizar strings (lowercase, strip, encoding UTF-8)')
print(f'   [x] Criar coluna text_field = {" + ".join(TEXT_COLS)}')
print(f'   [x] Exportar songs_clean.csv')
print('=' * 60)

In [ ]:
# Exportar relatório de metadados para o grupo
meta = pd.DataFrame({
    'coluna'       : df.columns,
    'tipo'         : df.dtypes.values,
    'nulos'        : df.isnull().sum().values,
    'nulos_%'      : (df.isnull().sum().values / total * 100).round(2),
    'únicos'       : df.nunique().values,
    'min'          : df.min(numeric_only=True).reindex(df.columns).values,
    'max'          : df.max(numeric_only=True).reindex(df.columns).values,
    'média'        : df.mean(numeric_only=True).reindex(df.columns).round(3).values,
})
meta.to_csv('../data/relatorio_eda.csv', index=False)
print('Relatório salvo em: data/relatorio_eda.csv')